
# Procesamiento de la Carta Marina de Córdoba 2013

## Descripción

Este notebook tiene como objetivo reconstruir y documentar el proceso de extracción de datos de la Carta Marina de Córdoba correspondiente a las elecciones de 2015.

El desarrollo se basa en el trabajo realizado en el repositorio **Carta Marina Córdoba 2017** de avdata99 (https://github.com/avdata99/carta-marina-2017/tree/master).

En esta primera etapa el trabajo se concentra exclusivamente en la Carta Marina 2013.

## Flujo de trabajo


1. PDF
2. TXT (pdftotext -layout)
3. CSV de mesas/escuelas/electores


**Productos esperados**

- `LugaresDeVotacion-elecciones-2013.pdf`
- `carta-marina-cordoba-2013.txt`
- `escuelas-elecciones-2013-cordoba.csv`



0. Dependencias

In [1]:
from google.colab import files
import pandas as pd
from pathlib import Path

In [2]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up libpoppler118:amd64 (22.02.0-2ubuntu0.13) ...
Setting up poppler-

#1. Carga del PDF

In [3]:
archivo = files.upload()

nombre_archivo = next(iter(archivo))

print(f"Archivo cargado: {nombre_archivo}")


Saving 2013.pdf to 2013.pdf
Archivo cargado: 2013.pdf


#2. PDF a TXT

In [4]:
nombre_txt = Path(nombre_archivo).with_suffix(".txt")

!pdftotext -layout "{nombre_archivo}" "{nombre_txt}"

print(f"Archivo generado: {nombre_txt}")

Archivo generado: 2013.txt


#3. TXT a CSV



```
Leer línea
    │
    ├── ¿Es un encabezado o número de página?
    │      └── Sí → ignorar
    │
    ├── ¿Es una sección?
    │      └── Sí → actualizar sección
    │
    ├── ¿Es un circuito?
    │      └── Sí → actualizar circuito e iniciar lectura de establecimientos
    │
    ├── ¿Es el resumen del circuito?
    │      └── Sí → finalizar lectura del circuito
    │
    ├── ¿Está vacía?
    │      └── Sí → ignorar
    │
    ├── ¿Estamos dentro de un circuito?
    │      └── No → continuar
    │
    └── Procesar establecimiento
           ├── Ignorar líneas "asociada a"
           ├── Reconstruir el nombre del establecimiento
           ├── Extraer mesas y electores
           ├── Validar continuidad de mesas
           └── Guardar registro
```





##Inspeccion del TXT

In [5]:
path = f"/content/{nombre_txt}"

with open(path, "r", encoding="utf-8") as f:
    raw = f.read()

lines = raw.splitlines()

print(f"Cantidad total de líneas: {len(lines)}")
print("\nPrimeras 10 líneas:\n")

for i, linea in enumerate(lines[:10], start=1):
    print(i, repr(linea))
print ("...")
print ("...")
print ("...")
print ("...")
for i, linea in enumerate(raw.splitlines()[-10:], start=len(raw.splitlines())-9):
    print(i, repr(linea))

Cantidad total de líneas: 5848

Primeras 10 líneas:

1 '                                Secretaría Electoral CORDOBA'
2 '                                Elecciones Primarias 11 de Agosto de 2013'
3 '                        Informe de Establecimientos, Mesas y Electores Habilitados'
4 ''
5 ''
6 ''
7 '                  CAPITAL'
8 '                                                                                        Mesa        Cant. de Cant. de'
9 '   1 - SECCIONAL PRIMERA                                                             Desde/Hasta     Mesas Electores'
10 ' CENTRO EDUC.NIVEL MEDIO ADULTO DEAN FUNES 417                                        0001 / 0007         7     2.408'
...
...
...
...
5839 ''
5840 '   Mesas Habilitadas:                                                                  7.986'
5841 ''
5842 '   Electores Habilitados:                                                      2.645.525'
5843 ''
5844 ''
5845 ''
5846 ''
5847 '  05/08/2013                            

In [6]:
print("Saltos de línea \\n:", raw.count("\n"))
print("Saltos de página \\f:", raw.count("\f"))

print("Con split('\\n'):", len(raw.split("\n")))
print("Con splitlines():", len(raw.splitlines()))

Saltos de línea \n: 5772
Saltos de página \f: 76
Con split('\n'): 5773
Con splitlines(): 5848


##Inicialización de variables

In [21]:
# =====================================
# Inicialización de variables del parser
# =====================================

# Sección electoral (equivale al departamento de la provincia)
seccion_nro = 0
seccion_name = ""
esperando_seccion = True

# Circuito electoral dentro de la sección.
# Puede contener letras (ej.: 4A, 4B, 4C), por eso se almacena como texto.
circuito_nro = ""
circuito_name = ""

# Estado actual del parser.
# Cuando vale "escuelas", las líneas leídas corresponden a establecimientos.
imin = ""

# Contador de líneas procesadas del archivo TXT.
cnt = 0

# Contador de errores (heredado del parser original).
errores = 0

# Lista donde se almacenan todos los establecimientos extraídos.
# Cada elemento es un diccionario con la información de un establecimiento.
escuelas = []

# Registra las discontinuidades detectadas en la numeración de mesas.
# Se utiliza como control de calidad, pero no interrumpe la ejecución.
discontinuidades = []

# Última mesa procesada.
# Permite verificar que la numeración de mesas sea continua.
last_mesa = 0

# ============================
# Variables de diagnóstico
# ============================

# Cantidad de números de página ignorados.
paginas_ignoradas = 0

# Cantidad de encabezados "DISTRITO CORDOBA" ignorados.
distritos_ignorados = 0

# Cantidad de encabezados "ELECCIONES 2015" ignorados.
elecciones_ignoradas = 0

# Cantidad de encabezados "Informe de Establecimientos" ignorados.
informes_ignorados = 0

# Almacena las filas que no pudieron procesarse correctamente,
# junto con la información necesaria para su revisión.
filas_con_error = []

print(f"Contador inicial: {cnt}")


Contador inicial: 0


##Ciclo for

In [22]:
import re

for linea in lines:
    cnt += 1

    # Limpiar espacios al principio y al final para las comparaciones
    linea_limpia = linea.strip()


    # =====================================
    # Ignorar encabezados y pies de página
    # =====================================

    # Ignorar número de página
    if "Página" in linea:
        paginas_ignoradas += 1
        continue

    # Ignorar encabezado de Secretaría Electoral
    if linea_limpia == "Secretaría Electoral CORDOBA":
        distritos_ignorados += 1
        continue

    # Ignorar encabezado de elecciones
    if "Elecciones" in linea:
        elecciones_ignoradas += 1
        continue

    # Ignorar encabezado del informe
    if "Informe de Establecimientos" in linea:
        informes_ignorados += 1
        continue


    # =====================================
    # Detectar final de sección
    # =====================================

    # En 2013 el final de cada sección se identifica explícitamente.
    # Después de esta línea, la próxima línea válida será
    # el nombre de la nueva sección.
    if linea_limpia.startswith("RESUMEN DE LA SECCION"):
        esperando_seccion = True
        imin = ""
        continue


    # =====================================
    # Ignorar líneas vacías
    # =====================================

    if linea_limpia == "":
        continue


    # =====================================
    # Detectar sección
    # =====================================

    # En 2013 no aparece "Sección 1 - CAPITAL".
    # El documento muestra solamente el nombre: CAPITAL, CALAMUCHITA, etc.
    if esperando_seccion:
        seccion_nro += 1
        seccion_name = linea_limpia

        esperando_seccion = False
        imin = ""

        print(f"Sección {seccion_nro}: {seccion_name}")
        continue


    # =====================================
    # Detectar final de circuito
    # =====================================

    # En 2013 aparece "Resumen" sin tilde.
    if linea_limpia.startswith("Resumen del Circuito"):
        imin = ""
        continue


    # =====================================
    # Ignorar encabezados de columnas
    # =====================================

    # En 2013 los títulos de las columnas ocupan líneas propias.
    if linea_limpia.startswith("Mesa"):
        continue

    if linea_limpia.startswith("Desde/Hasta"):
        continue

    if linea_limpia.startswith("Cant. de"):
        continue

    if linea_limpia in ["Mesas", "Electores"]:
        continue


    # =====================================
    # Detectar circuito
    # =====================================

    # Ejemplos:
    # 1 - SECCIONAL PRIMERA
    # 4A - PUEBLO LAS FLORES
    # 14Q - VILLA WARCALDE
    #
    # El código puede ser numérico o alfanumérico.
    match_circuito = re.match(
        r"^(\d+[A-Z]?)\s*-\s*(.+)$",
        linea_limpia
    )

    if match_circuito:
        circuito_nro = match_circuito.group(1)
        contenido_circuito = match_circuito.group(2).strip()
        partes_nombre = contenido_circuito.split("    ")
        circuito_name = partes_nombre[0].strip()


        imin = "escuelas"

        print(f"Circuito {circuito_nro}: {circuito_name}")
        continue


    # =====================================
    # Procesar establecimientos
    # =====================================

    if imin == "escuelas":

        partes = linea.split("    ")

        # Conservar solo fragmentos con contenido
        datos = [x.strip() for x in partes if x.strip() != ""]

        print(f"{cnt} -- {datos}")

        if len(datos) == 3:
          ultimos_datos = datos[-1].split()
          if len(ultimos_datos) == 2:
             datos = [
                 datos[0],
                 datos[1],
                 ultimos_datos[0],
                 ultimos_datos[1]
                 ]





        # =====================================
        # Extraer datos del establecimiento
        # =====================================

        # En 2013 el orden es:
        #
        # establecimiento | rango mesas | cantidad mesas | electores
        #
        # Ejemplo:
        #
        # CENTRO EDUC... | 0001 / 0007 | 7 | 2.408

        establecimiento = " ".join(datos[:-3])

        rango_mesas = datos[-3].split(" / ")
        mesa_desde = int(rango_mesas[0])
        mesa_hasta = int(rango_mesas[1])

        cant_mesas = int(datos[-2])

        cant_electores = int(datos[-1].replace(".", ""))


        # =====================================
        # Validar continuidad de mesas
        # =====================================

        if mesa_desde != last_mesa + 1:
            discontinuidades.append({
                "linea": cnt,
                "esperada": last_mesa + 1,
                "encontrada": mesa_desde,
                "establecimiento": establecimiento
            })

        last_mesa = mesa_hasta


        # =====================================
        # Construir registro
        # =====================================

        elem = {
            "seccion_nro": seccion_nro,
            "seccion_name": seccion_name,
            "circuito_nro": circuito_nro,
            "circuito_name": circuito_name,
            "establecimiento": establecimiento.replace(",", "."),
            "cant_mesas": cant_mesas,
            "desde": mesa_desde,
            "hasta": mesa_hasta,
            "electores": cant_electores,
        }

        escuelas.append(elem)

Sección 1: CAPITAL
Circuito 1: SECCIONAL PRIMERA
10 -- ['CENTRO EDUC.NIVEL MEDIO ADULTO DEAN FUNES 417', '0001 / 0007', '7', '2.408']
11 -- ['ESC NUESTRA SEÑORA DEL HUERTO BELGRANO 269', '0008 / 0019', '12', '4.128']
12 -- ['COL NAC DE MONSERRAT OBISPO TREJO 294', '0020 / 0036', '17', '5.840']
13 -- ['ESC SANTA TERESA DE JESUS OBISPO TREJO Y SANABRIA 160', '0037 / 0046', '10', '3.430']
Circuito 2: SECCIONAL SEGUNDA
20 -- ['ESC JUAN BAUTISTA ALBERDI GRAL PAZ 486', '0047 / 0063', '17', '5.850']
Circuito 3: SECCIONAL TERCERA
27 -- ['ESC JERONIMO LUIS DE CABRERA SANTA ROSA 650', '0064 / 0078', '15', '5.145']
28 -- ['IPEM N° 270 GRAL M BELGRANO DEAN FUNES 850', '0079 / 0089', '11', '3.773']
29 -- ['ESC SANTO TOMAS CASEROS 745', '0090 / 0097', '8', '2.744']
30 -- ['ESC NORMAL ALEJANDRO CARBO AV COLON 951', '0098 / 0107', '10', '3.430']
31 -- ['ESC MARIANO MORENO SANTA ROSA ESQ SANTA FE', '0108 / 0121', '14', '4.791']
Circuito 4: NUEVA CORDOBA
38 -- ['ENET N° 3 LEOPOLDO LUGONES ITUZAINGO 483'

In [23]:
print(discontinuidades)
print("Establecimientos:", len(escuelas))
print("Discontinuidades:", len(discontinuidades))
print("Última mesa:", last_mesa)
for d in discontinuidades:
    print(d)

[{'linea': 178, 'esperada': 469, 'encontrada': 478, 'establecimiento': 'ESC DGO F SARMIENTO LOPEZ Y PLANES 2253 B°SAN VICENTE'}, {'linea': 179, 'esperada': 491, 'encontrada': 469, 'establecimiento': 'ESC SANTA MARGARITA DE CORTONA LOPEZ Y PLANES 2936 B°SAN VICENTE'}, {'linea': 180, 'esperada': 478, 'encontrada': 491, 'establecimiento': 'ESC PTE RIVADAVIA AGUSTIN GARZON 1563 B°SAN VICENTE'}, {'linea': 196, 'esperada': 523, 'encontrada': 527, 'establecimiento': 'ESC.PTE. JUAN D.PERÓN MANZANA 47,B° CIUDAD EVITA'}, {'linea': 197, 'esperada': 535, 'encontrada': 523, 'establecimiento': 'ESCUELA HEROES DE MALVINAS CALLE PUBLICA S/N (CNO VILLA POSSE)'}, {'linea': 204, 'esperada': 527, 'encontrada': 535, 'establecimiento': 'ESC TENIENTE GRAL ARAMBURU GARIBALDI 710 B°ALTO GRAL PAZ'}, {'linea': 273, 'esperada': 637, 'encontrada': 643, 'establecimiento': 'ESC UNESCO JUAN B JUSTO/AV N SRA D LOS MILAGROS'}, {'linea': 274, 'esperada': 656, 'encontrada': 637, 'establecimiento': 'IPEM N° 22 JUAN FILLOY

In [24]:
filas_problematicas_2013 = []

for nro_linea, linea in enumerate(lines, start=1):

    linea_limpia = linea.strip()

    # Ignorar líneas vacías
    if linea_limpia == "":
        continue

    # Ignorar encabezados y pies
    if "Página" in linea:
        continue

    if linea_limpia == "Secretaría Electoral CORDOBA":
        continue

    if "Elecciones" in linea:
        continue

    if "Informe de Establecimientos" in linea:
        continue

    if linea_limpia.startswith("RESUMEN DE LA SECCION"):
        continue

    if linea_limpia.startswith("Resumen del Circuito"):
        continue

    # Ignorar encabezados de columnas
    if linea_limpia.startswith("Mesa"):
        continue

    if linea_limpia.startswith("Desde/Hasta"):
        continue

    if linea_limpia.startswith("Cant. de"):
        continue

    if linea_limpia in ["Mesas", "Electores"]:
        continue

    # Ignorar líneas que parecen circuitos
    if re.match(r"^(\d+[A-Z]?)\s*-\s*(.+)$", linea_limpia):
        continue

    # Analizar separación por cuatro espacios
    partes = linea.split("    ")
    datos = [x.strip() for x in partes if x.strip() != ""]

    # Una fila normal debería producir al menos 4 partes:
    # establecimiento | rango | cantidad mesas | electores
    if len(datos) < 4:
        filas_problematicas_2013.append({
            "linea": nro_linea,
            "texto_original": linea,
            "datos": datos
        })


print("Cantidad de filas problemáticas:", len(filas_problematicas_2013))

for fila in filas_problematicas_2013:
    print("-" * 80)
    print("Línea:", fila["linea"])
    print("Original:", repr(fila["texto_original"]))
    print("Datos:", fila["datos"])

Cantidad de filas problemáticas: 82
--------------------------------------------------------------------------------
Línea: 7
Original: '                  CAPITAL'
Datos: ['CAPITAL']
--------------------------------------------------------------------------------
Línea: 1194
Original: '                        Establecimientos Habilitados:                                   342'
Datos: ['Establecimientos Habilitados:', '342']
--------------------------------------------------------------------------------
Línea: 1197
Original: '                        Elecrores Habilitados:                                    1.045.948'
Datos: ['Elecrores Habilitados:', '1.045.948']
--------------------------------------------------------------------------------
Línea: 1200
Original: '               CALAMUCHITA'
Datos: ['CALAMUCHITA']
--------------------------------------------------------------------------------
Línea: 1432
Original: '                        Establecimientos Habilitados:                

# Diagnostico de filas problematicas por su nombre y domicilio
para despues limpar a mano

In [15]:
# ============================================================
# DIAGNÓSTICO INDEPENDIENTE DE FILAS PROBLEMÁTICAS
# No modifica las variables del parser definitivo
# ============================================================

modo_diag = ""
linea_diag = 0
ultima_mesa_diag = 0

filas_correctas_diag = 0
filas_problematicas = []
discontinuidades_diag = []

for linea in lines:
    linea_diag += 1

    # Ignorar encabezados y elementos que no son escuelas
    if "Página" in linea:
        continue

    if linea.strip() == "DISTRITO CORDOBA":
        continue

    if "ELECCIONES 2015" in linea:
        continue

    if "Informe de Establecimientos" in linea:
        continue

    if linea.startswith("Sección "):
        modo_diag = ""
        continue

    if linea.startswith(" Circuito"):
        modo_diag = "escuelas"
        continue

    if linea == "":
        continue

    if linea.startswith("Resúmen del Circuito"):
        modo_diag = ""
        continue

    # Solo analizar líneas que deberían contener escuelas
    if modo_diag == "escuelas":
        partes = linea.split("    ")
        datos = [x.strip() for x in partes if x.strip() != ""]

        try:
            # Se prueba exactamente la estructura del parser original
            escuela = datos[0]
            cant_mesas = int(datos[1])

            rango_mesas = datos[2].split(" a ")
            mesa_desde = int(rango_mesas[0])
            mesa_hasta = int(rango_mesas[1])

            cant_electores = int(datos[3].replace(".", ""))

            # Registrar discontinuidades, pero sin detener el diagnóstico
            if mesa_desde != ultima_mesa_diag + 1:
                discontinuidades_diag.append({
                    "linea": linea_diag,
                    "esperada": ultima_mesa_diag + 1,
                    "encontrada": mesa_desde,
                    "contenido": linea,
                })

            ultima_mesa_diag = mesa_hasta
            filas_correctas_diag += 1

        except Exception as error:
            filas_problematicas.append({
                "linea": linea_diag,
                "contenido": linea,
                "datos": datos,
                "error": str(error),
            })

            print(f"Fila problemática en línea {linea_diag}")
            print(f"Datos obtenidos: {datos}")
            print(f"Error: {error}")
            print("-" * 80)

            # Intentar recuperar solamente el rango de mesas para que
            # el diagnóstico pueda continuar con la numeración correcta
            for fragmento in datos:
                if " a " in fragmento:
                    try:
                        rango_auxiliar = fragmento.split(" a ")
                        mesa_desde_aux = int(rango_auxiliar[0])
                        mesa_hasta_aux = int(rango_auxiliar[1])

                        if mesa_desde_aux != ultima_mesa_diag + 1:
                            discontinuidades_diag.append({
                                "linea": linea_diag,
                                "esperada": ultima_mesa_diag + 1,
                                "encontrada": mesa_desde_aux,
                                "contenido": linea,
                            })

                        ultima_mesa_diag = mesa_hasta_aux
                        break

                    except ValueError:
                        pass

            continue

In [16]:
print("Filas procesadas correctamente:", filas_correctas_diag)
print("Filas problemáticas:", len(filas_problematicas))
print("Discontinuidades detectadas:", len(discontinuidades_diag))
print("Última mesa encontrada:", ultima_mesa_diag)

Filas procesadas correctamente: 0
Filas problemáticas: 0
Discontinuidades detectadas: 0
Última mesa encontrada: 0


In [17]:
for fila in filas_problematicas:
    print("Línea:", fila["linea"])
    print("Contenido:", fila["contenido"])
    print("Separación obtenida:", fila["datos"])
    print("Error:", fila["error"])
    print("-" * 100)

#Dataframe

In [25]:
df_escuelas = pd.DataFrame(escuelas)

display(df_escuelas.head())
display(df_escuelas.tail())

print(df_escuelas.shape)

,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
0,1,CAPITAL,1,SECCIONAL PRIMERA,CENTRO EDUC.NIVEL MEDIO ADULTO DEAN FUNES 417,7,1,7,2408
1,1,CAPITAL,1,SECCIONAL PRIMERA,ESC NUESTRA SEÑORA DEL HUERTO BELGRANO 269,12,8,19,4128
2,1,CAPITAL,1,SECCIONAL PRIMERA,COL NAC DE MONSERRAT OBISPO TREJO 294,17,20,36,5840
3,1,CAPITAL,1,SECCIONAL PRIMERA,ESC SANTA TERESA DE JESUS OBISPO TREJO Y SANAB...,10,37,46,3430
4,1,CAPITAL,2,SECCIONAL SEGUNDA,ESC JUAN BAUTISTA ALBERDI GRAL PAZ 486,17,47,63,5850


,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
1158,26,Establecimientos Habilitados: ...,397A,CORRAL DEL BAJO,ESCUELA VALENTIN ALSINA CORRAL DEL BAJO,1,7972,7972,61
1159,26,Establecimientos Habilitados: ...,400,SAN MARCOS SUD,INST JOSE DE SAN MARTIN SAN MARCOS SUD,4,7973,7976,1301
1160,26,Establecimientos Habilitados: ...,400,SAN MARCOS SUD,ESC PROV M BUCHARDO SAN MARCOS SUD,4,7977,7980,1300
1161,26,Establecimientos Habilitados: ...,401,SANTA MARIA,ESC PROV PAULA ALBARRACIN PUBLICA S/N,1,7981,7981,197
1162,26,Establecimientos Habilitados: ...,402,VIAMONTE,INSTITUTO JUAN B ALBERDI AVELLANEDA 182,5,7982,7986,1542


(1163, 9)


In [26]:
df_escuelas.sample(20, random_state=42)

,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
942,20,Establecimientos Habilitados: ...,283,CONCEPCION,ESC.PROV.M.MORENO TUCUMAN 483,5,6709,6713,1659
101,1,CAPITAL,9,ESCOBAR,INST ESCUTI NAZARET 3399 - B° LAS ROSAS,10,956,965,3370
772,14,Establecimientos Habilitados: ...,195,ATAHONA,ESC.PROV.J.DE SAN MARTIN PUBLICA S/N,1,5862,5862,204
670,12,Establecimientos Habilitados: ...,156,VILLA CARLOS PAZ,ESC CARLOS NICANDRO PAZ SAN MARTIN 255,17,5022,5038,5865
881,18,Establecimientos Habilitados: ...,260A,EL VOLCAN,ESC PROV JUAN P PRINGLES PUBLICA S/N,1,6374,6374,60
244,1,CAPITAL,12F,JOSE HERNANDEZ,COL.JOVENES ARGENTINOS BOTAFOGO 3448 B° J.ESPI...,7,2273,2279,2408
872,17,Establecimientos Habilitados: ...,252,LEVALLE,ESC DR VIRGILIO BARBALATO CARLOS PELLEGRINI(B°...,8,6343,6350,2648
49,1,CAPITAL,5G,SAN VICENTE,ESC MARIA IGNACIA NAVARRO DE L CORRIENTES 2006...,9,502,510,3042
371,2,Establecimientos Habilitados: ...,25B,LOS MOLINOS,ESC PROV JUAN LARREA PUBLICA S/N,1,3188,3188,288
242,1,CAPITAL,12F,JOSE HERNANDEZ,ESC GRAL MANUEL BELGRANO OLAEN Y AMBOY - B° OÑA,10,2255,2264,3440


In [28]:
output_path = "escuelas-elecciones-2013-cordoba.csv"

df_escuelas.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo exportado: {output_path}")

Archivo exportado: escuelas-elecciones-2013-cordoba.csv
